In [2]:
import os
os.chdir(r'C:\Users\johnpaul\fraudguard-africa')
print(os.getcwd())

C:\Users\johnpaul\fraudguard-africa


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ====================== LOAD ENGINEERED DATA ======================
print("Loading engineered data...\n")
df = pd.read_csv('data/PS_20174392719_1491204439457_log.csv')

Loading engineered data...



In [4]:
# Re-apply feature engineering (for consistency)
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['oldbalanceDest'] - df['newbalanceDest']
df['amount_to_oldbalance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
df['amount_to_newbalance_ratio'] = df['amount'] / (df['newbalanceOrig'] + 1)

df['hour'] = df['step'] % 24
df['is_night'] = df['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)

In [5]:
# One-hot encoding
df = pd.get_dummies(df, columns=['type'], prefix='type', drop_first=True)

# Customer frequency
orig_freq = df['nameOrig'].value_counts()
df['orig_transaction_freq'] = df['nameOrig'].map(orig_freq)

df['high_risk_transaction'] = ((df['type_CASH_OUT'] == 1) & (df['amount'] > 100000)).astype(int)

print("Shape after features:", df.shape)

Shape after features: (6362620, 22)


In [6]:
# ====================== SELECT FEATURES ======================
# Drop columns we won't use for modeling
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud', 'step']
# Keep 'hour' for now

feature_cols = [col for col in df.columns if col not in drop_cols + ['isFraud']]

X = df[feature_cols]
y = df['isFraud']

print(f"Features used: {len(feature_cols)}")
print("Target shape:", y.shape)

Features used: 17
Target shape: (6362620,)


In [7]:
# ====================== TRAIN TEST SPLIT ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y   # Very important for imbalanced data
)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Testing samples: {X_test.shape[0]:,}")
print(f"Fraud in train: {y_train.sum()} ({y_train.mean()*100:.4f}%)")
print(f"Fraud in test: {y_test.sum()} ({y_test.mean()*100:.4f}%)")

Training samples: 5,090,096
Testing samples: 1,272,524
Fraud in train: 6570 (0.1291%)
Fraud in test: 1643 (0.1291%)


In [8]:
# ====================== FEATURE SCALING ======================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier viewing
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print("\nData preparation completed successfully!")


Data preparation completed successfully!
